# Week 4 L1 — Activity 3a: Transforming data

Cleaning made the data **correct**. Transforming makes it **useful**: you add new columns, change what a row means, and change the shape of the table so it fits the next job.

We use the clean health activity file you saved in Activity 1, `community_health_activity_clean.csv`. It is small enough to see every row, so you can check each result with your own eyes.

You will learn five transformations:

| # | Transformation | What it does |
|---|---|---|
| 1 | Calculate a new column | make a new number from existing numbers |
| 2 | Split a text column | pull numbers out of text like `"18-64"` |
| 3 | Put numbers into bands | turn a number into a category: Short / Medium / Long |
| 4 | Change the grain | add up many rows into fewer, bigger rows |
| 5 | Wide ↔ long | turn columns into rows, and rows into columns |

Run each cell with **Shift + Enter**. Guess the result before you run it.

In [1]:
import pandas as pd
import numpy as np

activity = pd.read_csv("community_health_activity_clean.csv")
activity["appointments"] = activity["appointments"].astype("Int64")
activity

,area_code,area_name,reporting_year,age_group,sex,appointments,average_wait_days,follow_up_needed,notes
0,CH001,Northside,2024,0-17,Female,1250,7.0,Yes,school clinics
1,CH001,Northside,2024,18-64,Male,980,5.0,No,NaN
2,CH001,Northside,2024,65+,Female,430,7.5,Yes,telephone follow-up
3,CH001,Northside,2025,0-17,Female,1280,6.0,No,NaN
4,CH002,Riverside,2024,0-17,Female,875,6.0,No,NaN
5,CH002,Riverside,2024,18-64,Male,1040,8.0,Yes,evening service
6,CH002,Riverside,2024,65+,Male,310,7.5,No,data-entry error in wait
7,CH002,Riverside,2025,18-64,Unknown,1060,7.0,Unknown,sex and follow-up missing
8,CH003,Hilltown,2024,0-17,Female,620,9.0,Yes,NaN
9,CH003,Hilltown,2024,18-64,Male,900,10.0,No,NaN


Remember the grain from Activity 1: **one row = one area × one reporting year × one age group × one sex**.

The second line is needed because a CSV file only stores text. When it is read back, pandas sees a blank in `appointments` and uses decimals again, so we set the whole-number type `Int64` once more (*Activity 1, Step 5a*).

---
## 1. Calculate a new column

You can do arithmetic on a whole column at once. Writing the result to a new column name **adds** that column.

Waiting times are in days. Here we also show them in weeks:

In [3]:
activity["average_wait_weeks"] = activity["average_wait_days"] / 7
activity["average_wait_weeks"] = activity["average_wait_weeks"].round(1)

activity[["area_code", "age_group", "average_wait_days", "average_wait_weeks"]]

,area_code,age_group,average_wait_days,average_wait_weeks
0,CH001,0-17,7.0,1.0
1,CH001,18-64,5.0,0.7
2,CH001,65+,7.5,1.1
3,CH001,0-17,6.0,0.9
4,CH002,0-17,6.0,0.9
5,CH002,18-64,8.0,1.1
6,CH002,65+,7.5,1.1
7,CH002,18-64,7.0,1.0
8,CH003,0-17,9.0,1.3
9,CH003,18-64,10.0,1.4


The original column is untouched. Transformations **add** information; they should not quietly overwrite the data you started from.

> **Try it.** Add a column `appointments_thousands` that shows appointments in thousands, rounded to 2 decimal places. What happens in the row where `appointments` is missing?

In [4]:
activity["appointments_thousands"] = activity["appointments"] / 1000
activity["appointments_thousands"] = activity["appointments_thousands"].round(2)

activity[["area_code", "age_group", "average_wait_days", "average_wait_weeks","appointments_thousands"]]

,area_code,age_group,average_wait_days,average_wait_weeks,appointments_thousands
0,CH001,0-17,7.0,1.0,1.25
1,CH001,18-64,5.0,0.7,0.98
2,CH001,65+,7.5,1.1,0.43
3,CH001,0-17,6.0,0.9,1.28
4,CH002,0-17,6.0,0.9,0.88
5,CH002,18-64,8.0,1.1,1.04
6,CH002,65+,7.5,1.1,0.31
7,CH002,18-64,7.0,1.0,1.06
8,CH003,0-17,9.0,1.3,0.62
9,CH003,18-64,10.0,1.4,0.9


---
## 2. Split a text column

`age_group` holds two numbers hidden inside text: `"18-64"` means *from 18 to 64*. To sort or compare ages, we need them as real numbers.

`.str.split("-", expand=True)` cuts each value at the `-` and puts each piece in its own column. Column `0` is the part before the dash, column `1` the part after.

In [5]:
age_parts = activity["age_group"].str.split("-", expand=True)
age_parts

,0,1
0,0,17
1,18,64
2,65+,NaN
3,0,17
4,0,17
5,18,64
6,65+,NaN
7,18,64
8,0,17
9,18,64


Look at the `65+` rows: there is no dash, so column `1` is empty and column `0` still says `65+`.

We remove the `+` first, then split, then turn each piece into a number (*Activity 1, Step 5a*):

In [6]:
age_text = activity["age_group"].str.replace("+", "", regex=False)
age_parts = age_text.str.split("-", expand=True)

activity["age_from"] = pd.to_numeric(age_parts[0]).astype("Int64")
activity["age_to"] = pd.to_numeric(age_parts[1]).astype("Int64")

activity[["age_group", "age_from", "age_to"]].drop_duplicates()

,age_group,age_from,age_to
0,0-17,0,17
1,18-64,18,64
2,65+,65,<NA>


`65+` has no upper age, so `age_to` is honestly missing rather than invented.

You will use exactly this idea on real data: a period such as `"2001-03"` hides a start year.

---
## 3. Put numbers into bands

Sometimes a category is easier to read and group by than a raw number. `pd.cut()` sorts each number into a band.

- `bins` are the edges of the bands. `[0, 6, 9, np.inf]` makes three bands: **0–6**, **6–9** and **9 and above** (`np.inf` means "no upper limit").
- `labels` names each band, in the same order.
- A value equal to an edge goes into the band **below** it: 6.0 is "Short".

In [7]:
wait_bins = [0, 6, 9, np.inf]
wait_labels = ["Short", "Medium", "Long"]

activity["wait_band"] = pd.cut(activity["average_wait_days"], bins=wait_bins, labels=wait_labels)

activity[["area_code", "age_group", "average_wait_days", "wait_band"]]

,area_code,age_group,average_wait_days,wait_band
0,CH001,0-17,7.0,Medium
1,CH001,18-64,5.0,Short
2,CH001,65+,7.5,Medium
3,CH001,0-17,6.0,Short
4,CH002,0-17,6.0,Short
5,CH002,18-64,8.0,Medium
6,CH002,65+,7.5,Medium
7,CH002,18-64,7.0,Medium
8,CH003,0-17,9.0,Medium
9,CH003,18-64,10.0,Long


In [8]:
activity["wait_band"].value_counts()

wait_band
Medium    6
Short     3
Long      3
Name: count, dtype: int64

Where you put the edges is a **decision**, and it changes the counts. Always state your edges and why you chose them.

> **Try it.** Change the edges to `[0, 7, 10, np.inf]` and run the two cells again. How many rows change band?

---
## 4. Change the grain: add up rows

Our table has one row per area × year × age group × sex. Suppose we only need **total appointments per area per year**. Then several rows must be combined into one.

`.groupby(columns)` gathers rows that share the same values in those columns. A calculation after it, such as `.sum()`, runs **once per group**.

In [9]:
appointments_by_area_year = (
    activity.groupby(["area_code", "reporting_year"])["appointments"].sum()
)
appointments_by_area_year

area_code  reporting_year
CH001      2024              2660
           2025              1280
CH002      2024              2225
           2025              1060
CH003      2024              1795
           2025                 0
Name: appointments, dtype: Int64

Look at **CH003, 2025**. The total is `0`, but nobody counted zero appointments: the only row for that group had a **missing** count. By default `.sum()` skips missing values, and the sum of nothing is 0.

That is a silent error. `min_count=1` says "only give a total if there is at least one real number; otherwise leave it missing":

In [10]:
appointments_by_area_year = (
    activity.groupby(["area_code", "reporting_year"])["appointments"].sum(min_count=1)
)
appointments_by_area_year

area_code  reporting_year
CH001      2024              2660
           2025              1280
CH002      2024              2225
           2025              1060
CH003      2024              1795
           2025              <NA>
Name: appointments, dtype: Int64

The result has the group columns as its row labels. `.reset_index()` turns them back into ordinary columns, so we have a normal table again:

In [11]:
area_year = appointments_by_area_year.reset_index()
area_year = area_year.rename(columns={"appointments": "total_appointments"})
area_year

,area_code,reporting_year,total_appointments
0,CH001,2024,2660
1,CH001,2025,1280
2,CH002,2024,2225
3,CH002,2025,1060
4,CH003,2024,1795
5,CH003,2025,<NA>


The grain is now **one row = one area × one year**. Rows went from 12 to 6.

> **Question.** We lost the age group and sex columns. Why must they go when we add up to area × year?

*Your answer:*

---
## 5. Wide and long

The same numbers can be laid out two ways.

**Long**: one row per area per year. The year is a *value* in a column.

```
area_code  reporting_year  total_appointments
CH001      2024            2660
CH001      2025            1280
```

**Wide**: one row per area. Each year is its *own column*.

```
area_code  2024  2025
CH001      2660  1280
```

Wide is easy for people to read. Long is what computers usually want: it is easy to filter by year, to add a new year (a new row, not a new column), and to match with another table.

### Long → wide with `.pivot()`

- `index` = what each row should be
- `columns` = which column's values become the new column names
- `values` = what fills the cells

In [ ]:
wide = area_year.pivot(index="area_code", columns="reporting_year", values="total_appointments")
wide = wide.reset_index()
wide

### Wide → long with `.melt()`

`.melt()` does the reverse:

- `id_vars` = the columns to keep as they are (they identify the row)
- every other column is "melted" into two new columns: `var_name` holds the old column **name**, `value_name` holds the **value**

In [ ]:
long = wide.melt(id_vars=["area_code"], var_name="reporting_year", value_name="total_appointments")
long = long.sort_values(["area_code", "reporting_year"]).reset_index(drop=True)
long

We are back where we started: 6 rows, one per area per year.

> **Question.** The GDHI file from Activity 2 has one row per area and one column per year. Is it wide or long? Which function would turn it into one row per area per year?

*Your answer:*

---
## What you used today

| Tool | What it does |
|---|---|
| `df["new"] = df["a"] / 7` | calculate a new column |
| `.round(1)` | round to 1 decimal place |
| `.str.split("-", expand=True)` | split text into separate columns |
| `pd.cut(column, bins=..., labels=...)` | put numbers into named bands |
| `.groupby([...])["col"].sum(min_count=1)` | add up rows per group, keeping all-missing groups missing |
| `.reset_index()` | turn group labels back into ordinary columns |
| `.pivot(index=..., columns=..., values=...)` | long → wide |
| `.melt(id_vars=..., var_name=..., value_name=...)` | wide → long |

## Exit ticket

1. Which transformation changed the **grain** of the table? What was the grain before and after?
2. Why did `min_count=1` matter for CH003 in 2025?
3. Give one reason to store data **long** rather than wide.